<a href="https://colab.research.google.com/github/chiragbulbule/VaultInfer/blob/main/FinalTenSEALModel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%%writefile sentence_embedding.py

# This is a placeholder file for sentence_embedding.py
# You should replace its content with your actual code if you have it.

# Example placeholder for embedding and user_embed (assuming they are lists/arrays)
embedding = []
user_embed = []

# If your actual file has functions or classes, you would define them here:
# def some_function():
#     pass

In [ ]:
%%writefile dataset.py

# This is a placeholder file for dataset.py
# You should replace its content with your actual code if you have it.

# Example placeholder for train_labels (assuming it's a list)
train_labels = []

# If your actual file has other data or functions, you would define them here:
# def load_data():
#     pass

In [ ]:
import tenseal as ts
context=ts.context(
    ts.SCHEME_TYPE.CKKS,
    poly_modulus_degree=8192,
    coeff_mod_bit_sizes=[60,40,40,60]
)
context.global_scale=2**40
context.generate_galois_keys()
vector1=[]
vector2=[]
for i in range(0,10):
  vector1.append(i)
  vector2.append(i-5)
encrypt=ts.ckks_vector(context,vector1)
encrypt2=ts.ckks_vector(context,vector2)
dot=encrypt.dot(encrypt2)
print(dot)
decrypt=dot.decrypt()
print(decrypt)

In [ ]:
!pip install tenseal
import tenseal as ts

In [ ]:
!pip install sentence-transformers
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold,cross_val_score,cross_val_predict
from sentence_embedding import embedding,user_embed
from dataset import train_labels
import numpy as np
import joblib as jl

"""
This files contains the code for training and testing of the logistic regression model,along with its predictions for the given sentences(vector embeddings).

"""
#----------------------------------------------------------------------------Model Training-A-----------------------------------------------------------------#

# - Earlier used method

"""
This method used train_test_split where the dataset was divided into a 80/20 split - 80 for training and 20 for testing.
This method was abandoned because the results were not conclusive and varied on test basis.

x_train,x_test,y_train,y_test=train_test_split(embedding,train_labels,test_size=0.2,train_size=0.8,random_state=67,stratify=train_labels)
clf.fit(x_train,y_train)
accuracy=clf.score(x_test,y_test)
print(accuracy)

"""

# Currently used method

clf=LogisticRegression(max_iter=1000,class_weight="balanced",C=1)

#-----------------------------------------------------------------Cross_validation using StratifiedKFold--------------------------------------------------#

"""
This technique of cross validation is used to assess the accuracy of the model by splitting the data into 5 folds,
training and testing on each sentence to accurately assess accuracy and corresponding standard deviation of the model's accuracy.

"""

sk=StratifiedKFold(shuffle=True)

accuracy_cross_fold=cross_val_score(clf,embedding,train_labels,cv=sk)
print(f"Mean Accuracy of the model is : {accuracy_cross_fold.mean():0.2f}")
print(f"Standard deviation of the accuracy is : {np.std(accuracy_cross_fold):0.4f}")

prediction_cross_fold=cross_val_predict(clf,embedding,train_labels,cv=sk)
prediction_probability_cross_fold=cross_val_predict(clf,embedding,train_labels,cv=sk,method='predict_proba')

# print(prediction_cross_fold)
# print(prediction_probability_cross_fold)

#----------------------------------------------------------------------------Model Training-B-----------------------------------------------------------------#

"""
This is the code to actually train the model with the given dataset.

"""
clf.fit(embedding,train_labels)

#----------------------------------------------------------------------------Model Predictions-----------------------------------------------------------------#

# Accuracy of the model by inputting the same training data as the testing data - not an accurate measurement of accuracy

print(clf.score(embedding,train_labels))

# User input prediction

prediction=clf.predict(user_embed.reshape(1,384))
probability=clf.predict_proba(user_embed.reshape(1,384))

print(probability)
print(prediction)

# Test cases prediction

"""
prediction=clf.predict(test_cases_embed)
probability=clf.predict_proba(test_cases_embed)

print(probability)
print(prediction)

"""

#----------------------------------------------------------------SAVING MODEL,WEIGHTS AND BIAS TO RESPECTIVE FILES------------------------------------#

"""
weights=clf.coef_ #shape-(1,384)
bias=clf.intercept_ #shape-(1,)

flattened_weights=weights.flatten() #This is to make it compatible with tenseal parameters

np.save("./VaultInfer/Sentence_Classifier_model/New/vault_weights",flattened_weights)
np.save("./VaultInfer/Sentence_Classifier_model/New/vault_bias",bias)
jl.dump(clf,"./VaultInfer/Sentence_Classifier_model/New/vault_model",0)

"""

#----------------------------------------------------------------LOADING MODEL,WEIGHTS AND BIAS FROM RESPECTIVE FILES (TEST)------------------------------------#

"""
weights=np.load("./VaultInfer/Sentence_Classifier_model/New/vault_weights.npy")
bias=np.load("./VaultInfer/Sentence_Classifier_model/New/vault_bias.npy")

model=jl.load("./VaultInfer/Sentence_Classifier_model/New/vault_model")

print(weights[:4],weights.shape,bias)

# Testing if the model is loaded correctly

prediction=model.predict(user_embed.reshape(1,384))
probability=model.predict_proba(user_embed.reshape(1,384))

print(prediction,probability)

"""

#----------------------------------------------------------------MANUAL IMPLEMENTATION OF FORWARD PASS------------------------------------------------#

# Numpy version

"""
weighted_sum=np.dot(weights,test_embed) + bias

score=(0.5) + (weighted_sum/4) - (pow(weighted_sum,3)/48) + (pow(weighted_sum,5)/480) #-[0.81565414]-5th degree polynomial

"""

# Tenseal version

# To be completed

#--------------------------------------------------------------------------TEST CODE A---------------------------------------------------------------#

"""
predictions_all = clf.predict(embedding)
print(predictions_all)
print(type(predictions_all))


print(f"Predicted ALERT: {np.sum(predictions_all == 1)}")
print(f"Predicted NORMAL: {np.sum(predictions_all == 0)}")

test_sentences = [
    "Hydraulic pressure critical, shutdown imminent",
    "The meeting is at 3pm",
    "Unauthorized access detected at the server",
    "I'm making a cup of tea"
]
predictions = clf.predict(em)
probs=clf.predict_proba(em)
print(type(probs),probs)
print(type(predictions),predictions)

for sentence, prob in zip(test_sentences, probs):
    print(f"{prob[1]:.4f} — {sentence}")

prediction=clf.predict_proba(x_test)
for pre,actual in zip(prediction,y_test):
    print(pre,actual)

"""

#--------------------------------------------------------------------------TEST CODE B---------------------------------------------------------------#

"""

weighted_sum=np.dot(test_embed,weights) + bias #error because shapes don't meet requirements

print(np.dot(weights,test_embed)) # 1D Array

score_2=1/(1+np.exp(-weighted_sum)) #This uses sigmoid function-[0.81308656]
score_3=0.5+(0.15012*weighted_sum)-(0.001593*pow(weighted_sum,3)) #-[0.71564303] -3rd degree
score_4=0.5 + 0.197 * weighted_sum - 0.004 * weighted_sum **3 #-[ 0.777 ] -3rd degree


print(score,score_2,score_3,score_4)

all_weighted_sum=np.dot(weights,embedding.T) + bias
print(f"Min value is {all_weighted_sum.min():0.2f}")
print(f"Max value is {all_weighted_sum.max():0.2f}")
print(f"Mean value is {all_weighted_sum.mean():0.2f}")

"""

#--------------------------------------------------------------------------TEST CODE C---------------------------------------------------------------#

"""
Cross_validation using StratifiedKFold

index_list=[train_labels.index(true_label) for true_label,predict_label in zip(train_labels,prediction_cross_fold) if true_label != predict_label] - incorrect because it returns first appearance

index_list=[i for i,(actual_label,predict_label) in enumerate(zip(train_labels,prediction_cross_fold)) if actual_label!=predict_label]

wrongly_guessed_pairs=[(train_sentences[index],prediction_probability_cross_fold[index][1].item()) for index in index_list]
print(wrongly_guessed_pairs)

Testing which sentences are near the margin and which are way off

for i,predict_proba_tuple in enumerate(prediction_probability_cross_fold):
    if 0.4 <= predict_proba_tuple[1] <= 0.6:
        print(f"{train_sentences[i]} - {predict_proba_tuple[1]}")

    elif abs(predict_proba_tuple[1] - train_labels[i]) > 0.6:
        print(f"{train_sentences[i]} - {predict_proba_tuple[1]}")

"""

In [ ]:
import tenseal as ts
import numpy as np

#-------- LOAD AI ENGINEER'S WEIGHTS --------#
weights = np.load("vault_weights.npy")
bias = np.load("vault_bias.npy")

print(f"Weights shape: {weights.shape}")
print(f"Bias shape: {bias.shape}")
print(f"Bias value: {bias}")

In [ ]:
import tenseal as ts
import numpy as np
from sentence_transformers import SentenceTransformer

#-------- LOAD WEIGHTS --------#
weights = np.load("vault_weights.npy")
bias = np.load("vault_bias.npy")

#-------- LOAD EMBEDDING MODEL --------#
embedder = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
print("Embedding model loaded ✅")

#-------- TENSEAL CONTEXT --------#
def create_context():
    context = ts.context(
        ts.SCHEME_TYPE.CKKS,
        poly_modulus_degree=16384,
        coeff_mod_bit_sizes=[60, 40, 40, 40, 40, 60]
    )
    context.global_scale = 2**40
    context.generate_galois_keys()
    return context

context = create_context()
print("Encryption context ready ✅")

#-------- THE ACTUAL PIPELINE --------#
def classify_encrypted(sentence):
    print(f"\nInput: '{sentence}'")

    # Step 1 — Embedd
    user_vector = embedder.encode(sentence).tolist()
    print(f"Embedded into {len(user_vector)}")

    # Step 2 — Encrypt
    encrypted_vector = ts.ckks_vector(context, user_vector)
    print(f"Vector encrypted")
    # Step 3 — Encrypted forward pass (server side)
    encrypted_dot = encrypted_vector.dot(weights.tolist())
    encrypted_result = encrypted_dot + bias[0]
    print(f"Server computed on ciphertext ")

    # Step 4 — Decrypt
    decrypted_result = encrypted_result.decrypt()[0]

    # Step 5 — Sigmoid
    x = decrypted_result
    score = 0.5 + (x/4) - (x**3/48) + (x**5/480)
    score = max(0.0, min(1.0, score))  # clip to 0-1

    # Step 6 — Prediction
    prediction = "ALERT 🚨" if score >= 0.5 else "NORMAL ✅"
    print(f"Probability of ALERT: {score:.4f}")
    print(f"Prediction: {prediction}")
    return prediction

#-------- TEST WITH REAL SENTENCES --------#
classify_encrypted(input("Enter your prompt"))
